In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

X = np.array([
    [1.2, 0.7, -3.1, "blue"],
    [0.5, -1.0, 2.2, "red"],
    [-0.2, 4.1, 0.0, "red"],
    [3.3, 1.1, -0.9, "blue"],
    [1.0, 0.0, 1.5, "blue"]
], dtype=object)

y = np.array([0, 2, 1, 0])

model = models.Sequential([
    layers.Dense(16, activation="relu", input_shape=(5,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="relu")
    
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["mae"]
)

model.fit(
    X, y,
    epochs=10,
    batch_size=2
)

/Users/MAC/Desktop/HOF/ADL_PRACTICAL/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: Data cardinality is ambiguous. Make sure all arrays contain the same number of samples.'x' sizes: 5
'y' sizes: 4


In [2]:
# Correct version of code
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Original data
X_numeric = np.array([
    [1.2, 0.7, -3.1],
    [0.5, -1.0, 2.2],
    [-0.2, 4.1, 0.0],
    [3.3, 1.1, -0.9],
    [1.0, 0.0, 1.5]
])

X_color = np.array([
    ["blue"],
    ["red"],
    ["red"],
    ["blue"],
    ["blue"]
])

# One-hot encoding
encoder = OneHotEncoder(sparse_output=False)
X_color_encoded = encoder.fit_transform(X_color)

# Combine features
X = np.concatenate([X_numeric, X_color_encoded], axis=1)

# Correct labels
y = np.array([0, 2, 1, 0, 1])

# Build model
model = models.Sequential([
    layers.Dense(16, activation="relu", input_shape=(5,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="softmax")
])

# Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train
model.fit(
    X, y,
    epochs=10,
    batch_size=2
)

Epoch 1/10


/Users/MAC/Desktop/HOF/ADL_PRACTICAL/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2000 - loss: 1.3897  
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2000 - loss: 1.3606 
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2000 - loss: 1.3412 
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2000 - loss: 1.3180
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2000 - loss: 1.2972 
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2000 - loss: 1.2749 
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2000 - loss: 1.2560 
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2000 - loss: 1.2364
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2000 - loss: 1.2181     
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2000 - loss: 1.1995 


# Mistake 1: Number of Samples in `X` and `y` Do Not Match

## Why It Is Incorrect

The input dataset `X` contains 5 samples, while the target array `y` contains only 4 labels.

In supervised learning, every input sample must have exactly one corresponding output label. If the number of samples and labels are different, the model cannot correctly map inputs to outputs during training.

This causes TensorFlow to produce errors such as:

```python
ValueError: Data cardinality is ambiguous
```


# Mistake 2: Using String Values Directly in the Input Data

## Why It Is Incorrect

The dataset contains categorical string values such as `"blue"` and `"red"`.

Neural networks cannot directly process string data because Dense layers perform mathematical operations like matrix multiplication, which require numerical values.

Because of the mixed data types, NumPy converts the array into `dtype=object`, which is inefficient and incompatible with deep learning operations.

---

## How to Fix It

Convert categorical values into numerical form using encoding techniques such as One-Hot Encoding.

### Example

| Color | Encoded Form |
| ----- | ------------ |
| red   | [1, 0]       |
| blue  | [0, 1]       |

This converts the categorical feature into numerical values that the neural network can process correctly.


# Mistake 3: Incorrect `input_shape`

## Why It Is Incorrect

The model uses:

```python
input_shape=(5,)
```

This tells TensorFlow that every input sample contains **5 features**.

However, the original dataset actually contains only **4 features**:

```python
[1.2, 0.7, -3.1, "blue"]
```

These are:

- `1.2` → Numerical feature 1
- `0.7` → Numerical feature 2
- `-3.1` → Numerical feature 3
- `"blue"` → Categorical feature 4

So before preprocessing, the dataset has only **4 columns**, not 5.

The input layer of a neural network must exactly match the number of features in the dataset. If the shape does not match, TensorFlow may produce shape mismatch errors during training because the network structure and input data are inconsistent.

---

## How to Fix It

The correct `input_shape` depends on whether preprocessing has already been applied.

### Before Encoding

Before converting categorical values into numerical form, the dataset contains 4 features.

So the correct version is:

```python
input_shape=(4,)
```

---

### After One-Hot Encoding

If One-Hot Encoding is applied to the categorical feature (`"red"` and `"blue"`), the number of features increases.

Example:

| Original Value | Encoded Form |
| -------------- | ------------ |
| red            | [1, 0]       |
| blue           | [0, 1]       |

Now the dataset contains:

- 3 numerical features
- 2 encoded categorical columns

Total:

```text
5 features
```

So after preprocessing, the correct input shape becomes:

```python
input_shape=(5,)
```

The input shape should always match the final processed dataset used for training.


# Mistake 4: Wrong Activation Function in the Output Layer

## Why It Is Incorrect

The output layer uses:

```python
layers.Dense(3, activation="relu")
```

`ReLU` is commonly used in hidden layers because it helps neural networks learn complex patterns efficiently.

However, this is a **multi-class classification problem**, and the output layer should return probability values for each class.

ReLU is not suitable for this because:

- it only returns values between `0` and `∞`
- it does not produce probabilities
- the outputs do not sum to 1

Because of this, the model cannot correctly represent class probabilities.

For classification tasks, the output layer should indicate how likely each class is, which ReLU cannot properly provide.

---

## How to Fix It

Use the `softmax` activation function in the output layer.

### Correct Version

```python
layers.Dense(3, activation="softmax")
```

Softmax converts the output values into probabilities.

Example:

```text
[0.1, 0.7, 0.2]
```

This means:

- 10% probability for class 1
- 70% probability for class 2
- 20% probability for class 3

The probabilities always add up to 1, which makes Softmax the correct activation function for multi-class classification problems.


# Mistake 5: Using MAE as a Metric for Classification

## Why It Is Incorrect

The model uses:

```python
metrics=["mae"]
```

`MAE` (Mean Absolute Error) is mainly used for regression problems.

It measures the average numerical difference between predicted values and actual values.

However, this model is solving a classification problem, where the goal is to predict the correct class label rather than a continuous numerical value.

Because of this:

- MAE does not properly measure classification performance
- it does not show how many predictions are correct
- it is not meaningful for evaluating classification accuracy

Using MAE may therefore give misleading evaluation results.

---

## How to Fix It

Use `accuracy` as the evaluation metric.

### Correct Version

```python
metrics=["accuracy"]
```

Accuracy measures how many predictions were classified correctly.

Formula:

```text
Accuracy = Correct Predictions / Total Predictions
```

This makes accuracy the most suitable and commonly used metric for classification problems.


# Mistake 6: Using `dtype=object`

## Why It Is Incorrect

The dataset becomes:

```python
dtype=object
```

because it contains both numerical values and string values together.

Example:

```python
[1.2, 0.7, -3.1, "blue"]
```

NumPy cannot store numbers and strings together in a normal numerical array, so it converts the entire dataset into an object array.

This creates problems because TensorFlow and deep learning models work best with numerical tensor types such as:

```python
float32
int32
```

Using `dtype=object` can cause:

- slower computations
- inefficient memory usage
- tensor conversion problems
- training compatibility issues

Neural networks require fully numerical input data.

---

## How to Fix It

Convert all categorical values into numerical form before training.

One common solution is One-Hot Encoding.

After preprocessing, ensure the dataset contains only numerical values.

Example:

```python
X = X.astype(np.float32)
```

This converts the dataset into a TensorFlow-compatible numerical format that can be processed efficiently during training.


# Mistake 7: Categorical Data Was Not Properly Preprocessed

## Why It Is Incorrect

The dataset contains categorical values such as:

```python
"red"
"blue"
```

These values were directly passed into the neural network without preprocessing.

Neural networks cannot understand text or categorical labels directly because they perform mathematical operations only on numerical data.

Without proper preprocessing:

- the model cannot interpret categorical information correctly
- tensor conversion errors may occur
- training may fail
- model performance may become unstable

Preprocessing categorical data is an essential step in machine learning and deep learning.

---

## How to Fix It

Convert categorical values into numerical form before training the model.

A common method is **One-Hot Encoding**.

Example:

| Color | Encoded Form |
| ----- | ------------ |
| red   | [1, 0]       |
| blue  | [0, 1]       |

This converts categorical values into binary numerical vectors that the neural network can process correctly.

Example implementation:

```python
from sklearn.preprocessing import OneHotEncoder
```

After encoding, the dataset becomes fully numerical and suitable for TensorFlow training.


## 🔹 Task 2: Housing Price Prediction with Keras

Instructions:

- Download the housing_prices.csv dataset from Moodle
- Train an MLP to predict house sale prices

Requirements:

- Use all available features
- Encode categorical features appropriately
- Scale numerical features appropriately
- Use at least one hidden layer

Implementation
Use:

- Keras Functional API
  - https://keras.io/guides/functional_api/

Model Evaluation

- Choose 2 evaluation metrics
- Explain why you selected them

Data Split

- Use a 70/30 train-test split

📌 Note

- Carefully preprocess the dataset before training
- Handle categorical and numerical data correctly
- Compare model performance using the selected metrics
- Focus on understanding the full pipeline:
  - preprocessing
  - training
  - evaluation


In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense

# Load dataset
df = pd.read_csv("../data/housing_prices.csv")

# Features and target
X = df.drop("price", axis=1)
y = df["price"]

# Identify categorical and numerical columns
categorical_cols = ["mainroad", "neighborhood", "airconditioning"]
numerical_cols = ["area", "stories"]

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Input dimension
input_dim = X_train_processed.shape[1]

# Functional API model
inputs = Input(shape=(input_dim,))

x = Dense(64, activation="relu")(inputs)
x = Dense(32, activation="relu")(x)

outputs = Dense(1)(x)

model = Model(inputs=inputs, outputs=outputs)

# Compile model
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# Train model
history = model.fit(
    X_train_processed,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=4
)

# Evaluate model
test_loss, test_mae = model.evaluate(
    X_test_processed,
    y_test
)

print("Test Loss:", test_loss)
print("Test MAE:", test_mae)

Epoch 1/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 25708852150272.0000 - mae: 4760779.5000 - val_loss: 26268902883328.0000 - val_mae: 4768808.5000
Epoch 2/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 953us/step - loss: 25708569034752.0000 - mae: 4760750.5000 - val_loss: 26268315680768.0000 - val_mae: 4768749.5000
Epoch 3/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 25707319132160.0000 - mae: 4760623.0000 - val_loss: 26266199654400.0000 - val_mae: 4768535.0000
Epoch 4/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 952us/step - loss: 25703894482944.0000 - mae: 4760276.0000 - val_loss: 26261309095936.0000 - val_mae: 4768040.0000
Epoch 5/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 25697164722176.0000 - mae: 4759577.5000 - val_loss: 26252551389184.0000 - val_mae: 4767153.5000
Epoch 6/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 25686005776384.0000 - mae: 4758446.5000 - val_loss: 26239070896128.0000 - val_mae: 4765787.5000
Epoch 7/100
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 943us/step - loss: 25669